# recipe-dataclass — ex2: construct Recipe for binary add_forward — both parents, both unboxed args

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `recipe-dataclass`. Running the final beacon cell reports progress against the `Backprop: Recipe dataclass` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Recipe dataclass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`recipe-dataclass`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "recipe-dataclass"
DD_SUBTOPIC = "Backprop: Recipe dataclass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Recipe dataclass — quick refresher

A Recipe records exactly enough to replay a forward in reverse: `(func, args, kwargs, parents)` with `args` as the **unboxed raw arrays** and `parents` as a `{argidx: MiniTensor}` map.

**Worked exemplar.** A binary `add_forward(x, y)`:
```python
out = MiniTensor(x.array + y.array)
out.recipe = Recipe(
    func=t.add, args=(x.array, y.array), kwargs={},
    parents={0: x, 1: y},
)
```
Both inputs unboxed in `args`; both Tensors keyed by argidx in `parents`.

### Exercise 2 — construct Recipe for binary add_forward — both parents, both unboxed args

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the 4-field Recipe construction to a binary forward (add_forward): args = (x.array, y.array), parents = {0: x, 1: y}, with kwargs = {}.
> Keywords: recipe, binary-op, add-forward, two-parents
> ```

**KCs targeted:** `recipe-dataclass`, `recipe-args-unboxed-correctly`

Implement `add_forward(x, y)` for `MiniTensor` inputs:

1. Compute `out_raw = x.array + y.array`.
2. Return a NEW `MiniTensor` with `.array = out_raw` and a fully populated `.recipe`:
   - `recipe.func = t.add`
   - `recipe.args = (x.array, y.array)` — both inputs UNBOXED. Order matters: positional argnum 0 first.
   - `recipe.kwargs = {}`
   - `recipe.parents = {0: x, 1: y}` — both Tensors keyed by their argnum, referring to the ORIGINAL `MiniTensor` objects (not their `.array`).

**Key contrast with the unary case.** A unary `log_forward(x)` has `parents = {0: x}` — one entry. The binary case has two, both indexed. Skipping either parent would orphan a branch of the graph and break the reverse pass.

The test verifies field shape, numerical correctness, AND identity: `recipe.parents[0] is x` (the exact object, not a copy) — because the reverse pass mutates `.grad` on the parent.

In [ ]:
def add_forward(x, y):
    out = MiniTensor(x.array + y.array)
    out.recipe = Recipe(
        func=t.add,
        args=(x.array, y.array),
        kwargs={},
        parents={0: x, 1: y},
    )
    return out


<details><summary>Solution</summary>

```python
def add_forward(x, y):
    out = MiniTensor(x.array + y.array)
    out.recipe = Recipe(
        func=t.add,
        args=(x.array, y.array),
        kwargs={},
        parents={0: x, 1: y},
    )
    return out
```

**Two parents, not one.** A binary op like `add` has two inputs that both participate in the gradient — `d(x+y)/dx = 1` AND `d(x+y)/dy = 1`. Both must appear in `parents` so the reverse pass routes the gradient back to both. Omitting `1: y` would silently drop `y`'s gradient.

**`args` unboxes; `parents` does not.** `args` contains the raw tensors that get passed to the back fn — `add_back0(grad_out, out, x.array, y.array)`. `parents` keeps the MiniTensor identity so the reverse pass can write `parent.grad += grad_in`. Two different purposes, two different containers.

**Identity (`is`) vs equality (`==`).** The test uses `is` because the reverse pass mutates the parent's `.grad` attribute — a copy would receive the mutation in vain. PyTorch's autograd has the same invariant: `param.grad += ...` only updates the original `param`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()